# 05 · Refusal direction (RQ1)

Computes the refusal direction r_ℓ = mean(harmful) − mean(harmless) at every layer from the calibration pairs, picks the layer ℓ* whose ablation removes refusal best on the validation pairs, and checks it against random directions (Table A rows; docs/02 §3.1). **This is the riskiest experiment: if ablation does not beat random directions, stop and rethink.**

| | |
|---|---|
| **Kaggle settings** | Accelerator: GPU T4 · Internet: **On** · Secrets: `HF_TOKEN` |
| **Attach as inputs** | `safetypulse-lib` (sp_utils.py) · outputs of **03** and **04** |
| **Reads** | `pairs.jsonl`, `token_sets.json` |
| **Writes** (`/kaggle/working/`) | `direction/directions.safetensors`, `direction/refusal_direction.json` |

Keep this notebook and its outputs **private**: they contain harmful prompts.
After it finishes: *Save Version → Save & Run All*, then download the executed notebook into
`kaggle/executed/` and attach this notebook's output to the next notebook.

In [1]:
# Setup: find sp_utils.py (attach the `safetypulse-lib` dataset: see kaggle/README.md)
import glob, os, sys
_found = (glob.glob("/kaggle/input/**/sp_utils.py", recursive=True)
          + glob.glob("/kaggle/usr/lib/**/sp_utils.py", recursive=True)
          + glob.glob("../lib/sp_utils.py"))
assert _found, "sp_utils.py not found. Attach the safetypulse-lib dataset (Add Input) and re-run."
sys.path.insert(0, os.path.dirname(os.path.abspath(_found[0])))
import sp_utils as sp
sp.env_info()

{'torch': '2.11.0+cu130',
 'transformers': '5.20.2',
 'cuda_available': True,
 'on_kaggle': True,
 'smoke': False,
 'peft': '0.21.2',
 'gpus': ['Tesla T4', 'Tesla T4'],
 'capability': [7, 5],
 'vram_gb': 15.6}

In [2]:
import random
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from safetensors.torch import save_file

MODEL = os.environ.get("SP_MODEL", sp.DEFAULT_MODEL)
SEED = 0
N_RANDOM = 10 if not sp.SMOKE else 2
MAX_LAYER_FRAC = 0.8          # Arditi et al.: skip the last layers when choosing l*
GEN_TOKENS = 64
OUT = sp.out_dir("direction")
sp.set_seed(SEED)
pairs = sp.load_jsonl(sp.find_input("pairs.jsonl"))
token_sets = sp.load_json(sp.find_input("token_sets.json"))[MODEL]
calib = [p for p in pairs if p["split"] == "calib"]
val = [p for p in pairs if p["split"] == "val"]
print(len(calib), "calib", len(val), "val", "| matched pairs:", pairs[0]["matched"])
model, tok = sp.load_model(MODEL, dtype=torch.float32)

200 calib 50 val | matched pairs: False


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [3]:
acts_h = sp.resid_last(model, tok, [p["harmful"] for p in calib])
acts_s = sp.resid_last(model, tok, [p["harmless"] for p in calib])
directions = sp.mean_diff(acts_h, acts_s)                 # [resid, d_model]
L = sp.n_resid(model) - 1
candidates = list(range(1, int(MAX_LAYER_FRAC * L) + 1))
print("resid positions", directions.shape, "candidates", candidates[0], "...", candidates[-1])

OutOfMemoryError: CUDA out of memory. Tried to allocate 3.01 GiB. GPU 0 has a total capacity of 14.56 GiB of which 2.07 GiB is free. Including non-PyTorch memory, this process has 12.49 GiB memory in use. Of the allocated memory 12.22 GiB is allocated by PyTorch, and 153.52 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
val_h = [p["harmful"] for p in val]
val_s = [p["harmless"] for p in val]
base_h = float(sp.refusal_logit_diff(model, tok, val_h, token_sets).mean())
base_s = float(sp.refusal_logit_diff(model, tok, val_s, token_sets).mean())
scores = []
for i in sp.progress(candidates, desc="layers"):
    with sp.hooked(sp.ablate_direction(model, directions[i])):
        ablated = float(sp.refusal_logit_diff(model, tok, val_h, token_sets).mean())
    with sp.hooked(sp.add_direction(model, i, directions[i], 1.0)):
        induced = float(sp.refusal_logit_diff(model, tok, val_s, token_sets).mean())
    scores.append({"resid": i, "ablated_harmful": ablated, "induced_harmless": induced})
# l*: ablation lowers refusal on harmful prompts most, among layers where adding r also induces refusal
ok = [s for s in scores if s["induced_harmless"] > base_s] or scores
best = min(ok, key=lambda s: s["ablated_harmful"])
LAYER = best["resid"]
print(f"baseline harmful {base_h:.2f} / harmless {base_s:.2f}  ->  l* = resid {LAYER}", best)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot([s["resid"] for s in scores], [s["ablated_harmful"] for s in scores], label="harmful, r ablated")
ax.plot([s["resid"] for s in scores], [s["induced_harmless"] for s in scores], label="harmless, r added")
ax.axhline(base_h, ls="--", c="gray", lw=1, label="harmful, no intervention")
ax.axhline(base_s, ls=":", c="gray", lw=1, label="harmless, no intervention")
ax.axvline(LAYER, c="k", lw=0.8)
ax.set_xlabel("residual position (layer output)")
ax.set_ylabel("refusal logit difference")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "layer_selection.png", dpi=150)

## Table A rows: does ablating r beat ablating random directions? (generation-based refusal rate)

In [ ]:
def refusal_with(handles=None):
    with sp.hooked(handles or []):
        return sp.refusal_rate(sp.generate(model, tok, val_h, GEN_TOKENS, desc="val harmful"))

gen = torch.Generator().manual_seed(SEED)
rate_none = refusal_with()
rate_dir = refusal_with(sp.ablate_direction(model, directions[LAYER]))
rate_rand = [refusal_with(sp.ablate_direction(model, sp.random_unit_like(directions[LAYER], gen)))
             for _ in range(N_RANDOM)]
# Side effect check: does removing r make the model stop answering safe prompts? It should not.
with sp.hooked(sp.ablate_direction(model, directions[LAYER])):
    over_dir = sp.refusal_rate(sp.generate(model, tok, val_s, GEN_TOKENS, desc="val harmless"))
table_a = [
    {"ablation": "none", "refusal_rate": rate_none},
    {"ablation": f"refusal direction (resid {LAYER})", "refusal_rate": rate_dir},
    {"ablation": f"random directions (n={N_RANDOM})", "refusal_rate": float(np.mean(rate_rand)),
     "std": float(np.std(rate_rand))},
]
for r in table_a:
    r["delta_vs_none"] = r["refusal_rate"] - rate_none
pd.DataFrame(table_a)

In [ ]:
save_file({"directions": directions.contiguous()}, str(OUT / "directions.safetensors"))
sp.save_json({"model": MODEL, "layer": LAYER, "n_calib": len(calib), "n_val": len(val), "matched": pairs[0]["matched"],
              "baseline_logit_diff": {"harmful": base_h, "harmless": base_s}, "layer_scores": scores,
              "table_a": table_a, "random_rates": rate_rand, "harmless_refusal_with_r_ablated": over_dir,
              "token_sets": token_sets, "provenance": sp.provenance(SEED)}, OUT / "refusal_direction.json")
verdict = "PASS" if rate_dir < float(np.mean(rate_rand)) - 2 * float(np.std(rate_rand)) - 0.1 else "CHECK"
print(f"Gate G2 (direction part): {verdict}. Direction ablation {rate_dir:.2f} vs random {np.mean(rate_rand):.2f}")